### **Student Information**
GitHub ID: godfathern

Kaggle name: Pan

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

# **Project Report**

---

## 1. Model Development

**Summary:** The final submission is a soft-vote ensemble of **9 models** in two groups:
- 6 general-purpose models: logistic regressions on Qwen3 embeddings with and without TF-IDF, plus fine-tuned RoBERTa and MPNet.
- 3 transformers that were already trained on emotion data, fine-tuned again on our training set.

It scores a **5-fold cross-validated macro-F1 of 0.6418** on the training set (Kaggle public: 0.6285, private: 0.6547, rank 17). The baseline is 0.50.

### 1.1 Preprocessing Steps

* **Loading and joining.** The competition comes as three files, joined on `post_id`:
  * `data_identification.csv` says whether each post is train or test.
  * `final_posts.json` holds the text.
  * `emotion.csv` holds the label, for training posts only.

  After the join there are **3,360 training posts** and **1,440 test posts**.
* **Labels.** There are 6 classes, encoded in alphabetical order: anger, disgust, fear, joy, sadness, surprise. The classes are uneven:

  | joy | anger | sadness | disgust | surprise | fear |
  |---|---|---|---|---|---|
  | 875 | 674 | 628 | 454 | 441 | 288 |

Fear has about a third as many posts as joy, which is why the logistic regressions use balanced class weights (class_weight="balanced") and why the evaluation metric is macro-F1..
* **No text cleaning, on purpose.** The text is kept as it is. Capital letters, punctuation (`!!!`, `?`), emojis and placeholders such as `[NAME]` all carry emotion. The transformer tokenizers were pre-trained on raw text, so lowercasing or stripping symbols would only remove information. The only length limits are truncation to 128 tokens for the embedding models and 96 tokens for the fine-tuned models, which covers almost every post.
* **One validation scheme for everything.** A single `StratifiedKFold(5, shuffle=True, random_state=42)` split is created once and reused by **every** model. Each training post gets exactly one out-of-fold (OOF) prediction from a model that never saw it. That makes the CV scores of different models directly comparable, and lets the ensemble be scored without leakage. The test predictions of a model are the average of its 5 fold models.
* **Engineering setup.**
  * The notebook detects the device (CUDA, Apple MPS or CPU) and finds the data both locally and inside a Kaggle notebook.
  * Every finished model is cached in `ensemble_cache/`, so an interrupted run resumes where it stopped.
  * A model that crashes is skipped instead of stopping the whole run.

### 1.2 Feature Engineering Steps

Three kinds of features, one for each kind of model:

**a) Instruction-tuned sentence embeddings (Qwen3-Embedding-0.6B).**
* Qwen3-Embedding is steered by a task instruction written in front of the text. Every post is embedded **twice**, with two different instructions:
  * **A:** *"Classify the emotion expressed in the given text into one of six emotions: anger, disgust, fear, joy, sadness, surprise"*
  * **B:** *"Identify the main emotion that the writer of this social media post is feeling"*
* Each gives a 1024-dimensional, L2-normalised vector per post (shape 4800 × 1024 for train + test).
* The two instructions give two slightly different views of the same post, and their models make different mistakes, which helps the ensemble. Instruction A works better on its own (0.592 vs 0.576).
* The embeddings are computed in half precision and cached to disk.

**b) TF-IDF word and character n-grams.**
* Word unigrams and bigrams, plus character n-grams of 2–5 characters inside word boundaries (`char_wb`), with `min_df=2` and sublinear TF.
* The character n-grams catch spelling variants, elongations ("soooo"), slang and emoji sequences that a fixed vocabulary misses.
* **The TF-IDF is fitted inside each fold**, on the training part only, so the validation posts never influence the vocabulary or the IDF weights.
* It is stacked next to the dense embedding in one sparse matrix. Adding it raised the logistic regression from 0.592 to **0.613** (instruction A) and from 0.576 to **0.605** (instruction B). The surface wording adds information the embedding smooths away.

**c) Sub-word tokens for fine-tuning.**
* For the fine-tuned transformers, the input is the model's own sub-word token ids (maximum 96 tokens).
* To train faster, batches group posts of similar length (shuffle, then sort inside chunks of 20 batches) and are padded only to the next multiple of 16 tokens.
* The targets are **label-smoothed** (0.1): the correct class gets 0.917 and each other class 0.017. The labels are noisy (the same kind of text often appears with different emotions), and smoothing stops the model from becoming over-confident on doubtful labels.

### 1.3 Explanation of Your Model

**Step 1: Logistic regression on embeddings (4 models).**
* A multinomial logistic regression (`C=1`, `class_weight="balanced"`) is trained for each instruction (A, B), with and without the TF-IDF block.
* These models are fast, stable and add diversity to the ensemble. The frozen embedding already holds most of the meaning, so a linear model on top is enough.

**Step 2: Fine-tuned transformers (5 models).**

Each model gets a new 6-class output layer (`ignore_mismatched_sizes=True` replaces the old one) and is trained on each fold:
* Optimiser: AdamW, weight decay 0.01.
* Schedule: 10% linear warm-up, then linear decay.
* Batch size 16, 4 epochs.
* Loss: cross-entropy on the smoothed targets.
* Safety check: if a fold ends with a validation macro-F1 below 0.35 (the model never started learning), it is retrained with a new seed, up to 3 attempts.

The two groups:
* *General-purpose:* `FacebookAI/roberta-base` and `sentence-transformers/all-mpnet-base-v2`, at learning rate 3e-5.
* *Emotion-trained:* `SamLowe/roberta-base-go_emotions`, `j-hartmann/emotion-english-distilroberta-base` and `cardiffnlp/twitter-roberta-base-emotion-multilabel-latest`, at learning rate 2e-5. These checkpoints were already trained on public emotion datasets (GoEmotions, tweets), so they start from representations made for emotion. They needed a lower learning rate so as not to erase that. They are the three best single models.

**Step 3: Two-group soft-vote ensemble.**
* The class probabilities are averaged inside each group, then the two group averages are averaged.
* This way the 6 general models don't outvote the 3 emotion models just because there are more of them. Each kind of model has the same weight.
* The final label is the argmax.

**Results (5-fold CV macro-F1 on the 3,360 training posts):**

| Model | Group | CV macro-F1 |
|---|---|---|
| LogReg Qwen-0.6B-A | general | 0.5921 |
| LogReg Qwen-0.6B-A + TF-IDF | general | 0.6126 |
| LogReg Qwen-0.6B-B | general | 0.5762 |
| LogReg Qwen-0.6B-B + TF-IDF | general | 0.6052 |
| Fine-tuned roberta-base | general | 0.6030 |
| Fine-tuned all-mpnet-base-v2 | general | 0.5897 |
| Fine-tuned roberta-base-go_emotions | emotion | 0.6194 |
| Fine-tuned emotion-english-distilroberta-base | emotion | 0.6191 |
| Fine-tuned twitter-roberta-base-emotion-multilabel-latest | emotion | 0.6115 |
| **General-purpose group average** | | **0.6228** |
| **Emotion-trained group average** | | **0.6404** |
| **Final ensemble (9 models, 2 groups)** | | **0.6418** |

**Per-class results of the ensemble (out-of-fold):**

| Class | Precision | Recall | F1 |
|---|---|---|---|
| anger | 0.595 | 0.657 | 0.625 |
| disgust | 0.549 | 0.423 | **0.478** |
| fear | 0.687 | 0.625 | 0.655 |
| joy | 0.759 | 0.808 | **0.783** |
| sadness | 0.670 | 0.699 | 0.684 |
| surprise | 0.645 | 0.610 | 0.627 |
| **macro avg** | 0.651 | 0.637 | **0.642** |

Accuracy is 0.664.

**Observations:**
* Every single model ends up between 0.58 and 0.62. Ensembling different kinds of models adds about 2 points on top of the best one.
* **Joy** is the easiest class. **Disgust** is the hardest (recall 0.42): it is mostly predicted as anger, because the two use very similar words ("gross", "hate", insults).
* The class distribution of the predictions on the test set (joy 395, anger 339, sadness 264, surprise 190, disgust 139, fear 113) follows the training distribution, so the model is not collapsing onto one class.
* The same code has a larger configuration that switches on automatically on a CUDA GPU: Qwen3-Embedding-4B, multilingual-e5-large-instruct, roberta-large and deberta-v3-large. The final submission was made with the 9-model configuration above, on an Apple M2 (MPS).

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used an AI coding assistant during the competition. These are my main prompts, what it suggested, and what I kept.

1. **Prompt:** *"Here are the competition files (data_identification.csv, emotion.csv, final_posts.json). Load them, join them and give me a quick baseline for 6-class emotion classification scored with macro-F1."*
   **Suggestion:** Join on `post_id`, train a TF-IDF + logistic regression baseline, evaluate with stratified 5-fold CV instead of a single split.
   **Decision:** Kept. The 5-fold CV with one fixed seed became the scheme every later model uses, so all scores are comparable.

2. **Prompt:** *"Should I clean the text first, like lowercasing, removing punctuation, stopwords and emojis?"*
   **Suggestion:** Not for transformer models. Punctuation, capitals and emojis carry emotion, and the tokenizers expect raw text.
   **Decision:** Kept. No cleaning, only truncation.

3. **Prompt:** *"TF-IDF alone is weak. How can I use pretrained models without a big GPU?"*
   **Suggestion:** Use frozen sentence embeddings from an instruction-tuned embedding model (Qwen3-Embedding-0.6B) with a logistic regression on top. Write the emotion task into the instruction, and try two different instructions.
   **Decision:** Kept. Both instructions are used, because they give different errors.

4. **Prompt:** *"Can I combine the TF-IDF features with the embeddings? Make sure there is no data leakage."*
   **Suggestion:** Stack word and char n-gram TF-IDF next to the embedding in one sparse matrix, and fit the TF-IDF inside each fold.
   **Decision:** Kept. It gave about +2 to +3 points over the embedding alone.

5. **Prompt:** *"Write a fine-tuning loop for roberta-base that runs on my Mac (MPS), trains one model per fold and saves the out-of-fold predictions."*
   **Suggestion:** AdamW with warm-up and linear decay, length-grouped batches padded to a multiple of 16, label smoothing because the labels look noisy, and caching of finished models.
   **Decision:** Kept all of it. I also asked for a retry with a new seed when a fold doesn't learn, because one run got stuck predicting a single class.

6. **Prompt:** *"Every model I try stops around 0.60. What else can I do?"*
   **Suggestion:** Add models of a different kind and ensemble them, and try transformers that were already trained on emotion data (`SamLowe/roberta-base-go_emotions`, `j-hartmann/emotion-english-distilroberta-base`, `cardiffnlp/twitter-roberta-base-emotion-multilabel-latest`), fine-tuned again with a lower learning rate (2e-5).
   **Decision:** Kept. They are the three best single models (0.61–0.62).

7. **Prompt:** *"How should I combine the 9 models? Simple average or something smarter?"*
   **Suggestion:** Average the probabilities within each group, then average the two groups, so the six general models don't drown out the three emotion models. A stacking model was mentioned, but with only 3,360 posts it easily overfits the out-of-fold predictions.
   **Decision:** I used the two-group soft vote (0.6418 CV, against 0.6228 for the general group alone). I didn't use stacking.

8. **Prompt:** *"Make the code run on a Kaggle GPU too, with bigger models if a GPU is available, and organise it into preprocessing, feature engineering and model sections with comments."*
   **Suggestion:** Detect CUDA and switch to a large configuration (Qwen3-Embedding-4B, E5-large, roberta-large, deberta-v3-large). Check the Kaggle GPU and internet settings at start-up. Split the code into the three sections required by the report.
   **Decision:** The structure and comments are kept. The final submission uses the 9-model configuration.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
# =====================================================================================
# 1. PREPROCESSING
# Load the three competition files, join them on post_id and split into train / test.
# Also sets up what every later cell shares: the device, the 5 folds and the score function.
# The same code runs in two sizes: 9 models on a Mac (MPS) or CPU, and a larger set of
# models on a CUDA GPU (Kaggle notebook with "GPU T4 x2").
# =====================================================================================
import os, gc, glob, time, warnings, numpy as np, pandas as pd, scipy.sparse as sp
import torch, torch.nn.functional as F, transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
warnings.filterwarnings("ignore"); transformers.logging.set_verbosity_error()

found = glob.glob("/kaggle/input/**/data_identification.csv", recursive=True)        # inside a Kaggle notebook
ON_KAGGLE = bool(found)
DATA_DIR = os.path.dirname(found[0]) + "/" if ON_KAGGLE else "data/kaggle/2026-dm-lab-2-private-competition/"
ids = pd.read_csv(DATA_DIR + "data_identification.csv")      # post_id -> train / test
labels = pd.read_csv(DATA_DIR + "emotion.csv")                # post_id -> emotion (train only)
posts = pd.read_json(DATA_DIR + "final_posts.json")           # post_id -> text
df = ids.merge(posts, on="post_id").merge(labels, on="post_id", how="left")
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df = df[df["set"] == "test"].reset_index(drop=True)

classes = sorted(train_df["label"].unique())                  # 6 emotions, alphabetical
y = train_df["label"].map({c: i for i, c in enumerate(classes)}).values
n_train, n_test, n_cls = len(train_df), len(test_df), len(classes)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
BIG = device == "cuda"                                        # large configuration only on a CUDA GPU
CACHE = ("/kaggle/working/" if ON_KAGGLE else "./") + "ensemble_cache/"; os.makedirs(CACHE, exist_ok=True)
folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train_df, y))   # same folds for every model
macro_f1 = lambda p: f1_score(y, p.argmax(1), average="macro")
oof, test = {}, {}   # per model: out-of-fold probabilities (train) and fold-averaged probabilities (test)
print(f"train {n_train} | test {n_test} | device {device} | {'large' if BIG else 'small'} configuration")
if ON_KAGGLE:                                                 # the two usual setup mistakes, caught early
    import urllib.request
    assert BIG, "No GPU found. In the right sidebar set Accelerator to 'GPU T4 x2', then run again."
    try: urllib.request.urlopen("https://huggingface.co", timeout=15)
    except Exception: raise RuntimeError("No internet. In the right sidebar switch Internet on, then run again.")

def free_gpu():
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    elif device == "mps": torch.mps.empty_cache()

print(train_df["label"].value_counts().to_dict())
train_df.head()

train 3360 | test 1440 | device mps | small configuration
{'joy': 875, 'anger': 674, 'sadness': 628, 'disgust': 454, 'surprise': 441, 'fear': 288}


,post_id,set,text,label
0,f8bv5zh,train,Thought it looked weird as well.,disgust
1,q0rr1l9,train,Bear died in a pit,disgust
2,p94rwgk,train,I have endured worse.,sadness
3,9u6wtvm,train,The pottery around Brasov is in danger,fear
4,h8ea329,train,"Meanwhile, our soldiers were approaching the a...",fear


## 2. Feature Engineering Steps

In [2]:
# =====================================================================================
# 2. FEATURE ENGINEERING
# Three kinds of features, one per kind of model:
#   2a. sentence embeddings from frozen embedding models (computed here)
#   2b. TF-IDF word and character n-grams (function only: it is fitted inside each fold)
#   2c. sub-word token ids for the fine-tuned transformers (settings only)
# =====================================================================================
# 2a. Sentence embeddings from instruction-tuned embedding models. These models are steered by
#     a task instruction, so every post is embedded twice with two different instructions.
EMBEDDERS = {"Qwen/Qwen3-Embedding-0.6B": "Qwen-0.6B"}                                 # model -> short name
if BIG: EMBEDDERS = {"Qwen/Qwen3-Embedding-4B": "Qwen-4B", **EMBEDDERS, "intfloat/multilingual-e5-large-instruct": "E5-large"}
PROMPTS = {
    "A": "Instruct: Classify the emotion expressed in the given text into one of six emotions: anger, disgust, fear, joy, sadness, surprise\nQuery: ",
    "B": "Instruct: Identify the main emotion that the writer of this social media post is feeling\nQuery: ",
}
all_text = train_df["text"].tolist() + test_df["text"].tolist()

def embed(model_name):
    path = CACHE + "emb_" + model_name.split("/")[-1] + ".npz"
    if os.path.exists(path): return dict(np.load(path))
    t0 = time.time()
    embedder = SentenceTransformer(model_name, device=device,
                                   model_kwargs={"torch_dtype": torch.float16} if device != "cpu" else {})   # half precision: fits a 16 GB GPU
    embedder.max_seq_length = 128
    out = {}
    for key, prompt in PROMPTS.items():
        E = embedder.encode(all_text, prompt=prompt, batch_size=16 if BIG else 32, normalize_embeddings=True, show_progress_bar=False)
        out[key] = np.asarray(E, dtype=np.float32)
        if not np.isfinite(out[key]).all(): raise ValueError("embeddings contain NaN or infinity")
        print(f"  {model_name} prompt {key}: {out[key].shape} ({time.time() - t0:.0f}s)")
    del embedder; free_gpu()
    np.savez(path, **out)
    return out

embeddings = {}                                          # short name -> {"A": matrix, "B": matrix}, rows = train posts then test posts
for model_name, short in EMBEDDERS.items():
    try:
        embeddings[short] = embed(model_name)
    except Exception as e:
        print(f"!! {model_name} failed and is skipped: {type(e).__name__}: {str(e)[:300]}")
    free_gpu()
print({short: {key: E.shape for key, E in emb.items()} for short, emb in embeddings.items()})

# 2b. TF-IDF word and character n-grams. They are fitted inside each fold (see section 3a)
#     so that validation posts never influence the vocabulary or the IDF weights.
def tfidf_blocks(fit_text, *other_texts):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
    out = [sp.hstack([word.fit_transform(fit_text), char.fit_transform(fit_text)])]
    return out + [sp.hstack([word.transform(t), char.transform(t)]) for t in other_texts]

# 2c. For the fine-tuned transformers the "features" are sub-word token ids. Batches group
#     posts of similar length and pad to a multiple of 16 tokens, which keeps training fast.
MAX_LEN, BATCH, SMOOTH = 96, 16, 0.1

{'Qwen-0.6B': {'A': (4800, 1024), 'B': (4800, 1024)}}


## 3. Model Implementation Steps

In [3]:
# =====================================================================================
# 3. MODEL IMPLEMENTATION
#   3a. logistic regression on the embeddings, with and without TF-IDF
#   3b. fine-tuned transformers (general-purpose and emotion-trained)
#   3c. two-group soft-vote ensemble and the submission file
# Every model is scored on the same 5 folds. Finished models are cached in ensemble_cache/,
# so an interrupted run resumes where it stopped. Delete that folder to recompute everything.
# =====================================================================================
def run(name, fn):
    """Train one model (or load it from the cache). A failure skips that model instead of stopping the cell."""
    path = CACHE + "preds_" + name.replace("/", "_").replace(" ", "_").replace("+", "and") + ".npz"
    try:
        if os.path.exists(path):
            z = np.load(path); oof[name], test[name] = z["oof"], z["test"]
        else:
            oof[name], test[name] = fn(); np.savez(path, oof=oof[name], test=test[name])
        print(f"{name:38s} CV macro-F1 {macro_f1(oof[name]):.4f}")
    except Exception as e:
        oof.pop(name, None); test.pop(name, None)
        print(f"!! {name} failed and is skipped: {type(e).__name__}: {str(e)[:300]}")
    free_gpu()

# 3a. Logistic regression on the embeddings, with and without the TF-IDF block
def logreg_cv(E, use_tfidf):
    E_tr, E_te = E[:n_train], E[n_train:]
    o = np.zeros((n_train, n_cls), dtype=np.float32); t = np.zeros((n_test, n_cls), dtype=np.float32)
    for ia, ib in folds:
        A, B, T = E_tr[ia], E_tr[ib], E_te
        if use_tfidf:
            ta, tb, tt = tfidf_blocks(train_df["text"][ia], train_df["text"][ib], test_df["text"])
            A, B, T = [sp.hstack([s, sp.csr_matrix(d)]).tocsr() for s, d in ((ta, A), (tb, B), (tt, T))]
        clf = LogisticRegression(C=1, max_iter=3000, class_weight="balanced").fit(A, y[ia])   # balanced: classes are uneven
        o[ib] = clf.predict_proba(B); t += clf.predict_proba(T) / len(folds)
    return o, t

for short, emb in embeddings.items():
    for key in PROMPTS:
        for use_tfidf in (False, True):
            run(f"LogReg {short}-{key}" + (" + TF-IDF" if use_tfidf else ""), lambda E=emb[key], u=use_tfidf: logreg_cv(E, u))

# 3b. Fine-tuned transformers: one model per fold, each scored on its held-out fold
FINETUNE = [("FacebookAI/roberta-base", 3e-5, 4), ("sentence-transformers/all-mpnet-base-v2", 3e-5, 4)]   # (model, learning rate, epochs)
EMOTION = [("SamLowe/roberta-base-go_emotions", 2e-5, 4),                                  # already trained on public emotion datasets
           ("cardiffnlp/twitter-roberta-base-emotion-multilabel-latest", 2e-5, 4),
           ("j-hartmann/emotion-english-distilroberta-base", 2e-5, 4)]
FINETUNE += EMOTION
if BIG: FINETUNE += [("FacebookAI/roberta-large", 1.5e-5, 6), ("microsoft/deberta-v3-large", 1e-5, 6)]

def finetune_cv(model_name, lr, epochs):
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = lambda texts: tok(list(texts), truncation=True, max_length=MAX_LEN)["input_ids"]
    seq_tr, seq_te = enc(train_df["text"]), enc(test_df["text"])
    amp = device == "cuda"                               # mixed precision + gradient clipping on CUDA only

    def collate(seqs):                                   # pad a batch to a multiple of 16 tokens
        L = min(MAX_LEN, -(-max(len(s) for s in seqs) // 16) * 16)
        x = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long); m = torch.zeros((len(seqs), L), dtype=torch.long)
        for i, s in enumerate(seqs): x[i, :len(s)] = torch.tensor(s); m[i, :len(s)] = 1
        return x.to(device), m.to(device)

    def train_batches(idx, rng):                         # shuffle, then sort by length inside chunks of 20 batches
        idx = rng.permutation(idx); out = []
        for i in range(0, len(idx), BATCH * 20):
            chunk = sorted(idx[i:i + BATCH * 20], key=lambda j: len(seq_tr[j]))
            out += [chunk[k:k + BATCH] for k in range(0, len(chunk), BATCH)]
        return [out[i] for i in rng.permutation(len(out))]

    @torch.no_grad()
    def predict(model, seqs):
        model.eval(); order = np.argsort([len(s) for s in seqs]); out = np.zeros((len(seqs), n_cls), dtype=np.float32)
        for i in range(0, len(order), 64):
            b = order[i:i + 64]; x, m = collate([seqs[j] for j in b])
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                logits = model(input_ids=x, attention_mask=m).logits
            out[b] = F.softmax(logits.float(), -1).cpu().numpy()
        return out

    target = np.eye(n_cls, dtype=np.float32)[y] * (1 - SMOOTH) + SMOOTH / n_cls      # label smoothing: labels are noisy
    o = np.zeros((n_train, n_cls), dtype=np.float32); t = np.zeros((n_test, n_cls), dtype=np.float32); t0 = time.time()
    for f, (ia, ib) in enumerate(folds):
        for attempt in range(3):                         # a large model sometimes never starts learning: retry with a new seed
            torch.manual_seed(42 + f + 100 * attempt); rng = np.random.default_rng(42 + f + 100 * attempt)
            model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_cls, ignore_mismatched_sizes=True).to(device)   # replaces the old output layer
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01, fused=(device == "mps"))
            scaler = torch.amp.GradScaler("cuda", enabled=amp)
            steps = epochs * int(np.ceil(len(ia) / BATCH))
            sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)      # 10% warm-up, then linear decay
            for epoch in range(epochs):
                model.train()
                for b in train_batches(ia, rng):
                    x, m = collate([seq_tr[j] for j in b])
                    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                        logits = model(input_ids=x, attention_mask=m).logits
                    loss = -(torch.tensor(target[b], device=device) * F.log_softmax(logits.float(), -1)).sum(-1).mean()
                    scaler.scale(loss).backward()
                    if amp: scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
            val = predict(model, [seq_tr[j] for j in ib]); score = f1_score(y[ib], val.argmax(1), average="macro")
            if score >= 0.35 or attempt == 2: break
            print(f"  {model_name} fold {f + 1}/5 did not learn (val macro-F1 {score:.3f}), retrying with a new seed")
            del model, opt; free_gpu()
        o[ib] = val; t += predict(model, seq_te) / len(folds)
        print(f"  {model_name} fold {f + 1}/5: val macro-F1 {score:.4f} ({time.time() - t0:.0f}s)")
        del model, opt; free_gpu()
    return o, t

for model_name, lr, epochs in FINETUNE:
    run("Fine-tuned " + model_name.split("/")[-1], lambda n=model_name, l=lr, e=epochs: finetune_cv(n, l, e))

# 3c. Ensemble: soft vote in two steps. Average the general-purpose models, average the
#     emotion-trained models, then average the two groups so each kind has the same say.
cv = {name: macro_f1(p) for name, p in oof.items()}
emotion_names = ["Fine-tuned " + m.split("/")[-1] for m, _, _ in EMOTION]
groups = {"general-purpose": [n for n in oof if n not in emotion_names], "emotion-trained": [n for n in oof if n in emotion_names]}
groups = {g: names for g, names in groups.items() if names}
oof_ens = np.mean([np.mean([oof[n] for n in names], axis=0) for names in groups.values()], axis=0)
test_ens = np.mean([np.mean([test[n] for n in names], axis=0) for names in groups.values()], axis=0)
for g, names in groups.items():
    print(f"\n{g} group: CV macro-F1 {macro_f1(np.mean([oof[n] for n in names], axis=0)):.4f}")
    for n in sorted(names, key=cv.get, reverse=True): print(f"  {n:60s} {cv[n]:.4f}")
print(f"\nENSEMBLE of {len(oof)} models in {len(groups)} groups: CV macro-F1 {macro_f1(oof_ens):.4f}\n")
print(classification_report(y, oof_ens.argmax(1), target_names=classes, digits=3))

SUBMISSION = "/kaggle/working/submission.csv" if ON_KAGGLE else "submission_emotion.csv"
submission = pd.DataFrame({"id": test_df["post_id"], "label": [classes[i] for i in test_ens.argmax(1)]})
submission.to_csv(SUBMISSION, index=False)
print("saved", SUBMISSION, submission.shape, submission["label"].value_counts().to_dict())

LogReg Qwen-0.6B-A                     CV macro-F1 0.5921
LogReg Qwen-0.6B-A + TF-IDF            CV macro-F1 0.6126
LogReg Qwen-0.6B-B                     CV macro-F1 0.5762
LogReg Qwen-0.6B-B + TF-IDF            CV macro-F1 0.6052
Fine-tuned roberta-base                CV macro-F1 0.6030
Fine-tuned all-mpnet-base-v2           CV macro-F1 0.5897
Fine-tuned roberta-base-go_emotions    CV macro-F1 0.6194
Fine-tuned twitter-roberta-base-emotion-multilabel-latest CV macro-F1 0.6115
Fine-tuned emotion-english-distilroberta-base CV macro-F1 0.6191

general-purpose group: CV macro-F1 0.6228
  LogReg Qwen-0.6B-A + TF-IDF                                  0.6126
  LogReg Qwen-0.6B-B + TF-IDF                                  0.6052
  Fine-tuned roberta-base                                      0.6030
  LogReg Qwen-0.6B-A                                           0.5921
  Fine-tuned all-mpnet-base-v2                                 0.5897
  LogReg Qwen-0.6B-B                                       